In [2]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import json
import pandas as pd
import os
import tensorconvert as tcv
from tensorconvert import SecondOrderTensor, FourthOrderTensor

import yaml
import hashlib

In [13]:
dim = 2
FourthOrderTensor(dim=2, symmetry='major').as_voigt()


Matrix([
[a[0, 0, 0, 0], a[0, 0, 1, 1], a[0, 0, 0, 1]],
[a[0, 0, 1, 1], a[1, 1, 1, 1], a[1, 1, 0, 1]],
[a[0, 0, 0, 1], a[1, 1, 0, 1], a[0, 1, 0, 1]]])

In [26]:
FourthOrderTensor(dim=3, symmetry='major').as_voigt()

Matrix([
[a[0, 0, 0, 0], a[0, 0, 1, 1], a[0, 0, 2, 2], a[0, 0, 0, 1], a[0, 0, 0, 2], a[0, 0, 1, 2]],
[a[0, 0, 1, 1], a[1, 1, 1, 1], a[1, 1, 2, 2], a[1, 1, 0, 1], a[1, 1, 0, 2], a[1, 1, 1, 2]],
[a[0, 0, 2, 2], a[1, 1, 2, 2], a[2, 2, 2, 2], a[2, 2, 0, 1], a[2, 2, 0, 2], a[2, 2, 1, 2]],
[a[0, 0, 0, 1], a[1, 1, 0, 1], a[2, 2, 0, 1], a[0, 1, 0, 1], a[0, 1, 0, 2], a[0, 1, 1, 2]],
[a[0, 0, 0, 2], a[1, 1, 0, 2], a[2, 2, 0, 2], a[0, 1, 0, 2], a[0, 2, 0, 2], a[0, 2, 1, 2]],
[a[0, 0, 1, 2], a[1, 1, 1, 2], a[2, 2, 1, 2], a[0, 1, 1, 2], a[0, 2, 1, 2], a[1, 2, 1, 2]]])

In [16]:
x = sp.randMatrix(dim, dim)
x += x.T

x

Matrix([
[ 98, 174],
[174,  54]])

In [17]:
FourthOrderTensor(dim=dim).as_operator()(x)


Matrix([
[98*a[0, 0, 0, 0] + 348*a[0, 0, 0, 1] + 54*a[0, 0, 1, 1], 98*a[0, 0, 0, 1] + 348*a[0, 1, 0, 1] + 54*a[1, 1, 0, 1]],
[98*a[0, 0, 0, 1] + 348*a[0, 1, 0, 1] + 54*a[1, 1, 0, 1], 98*a[0, 0, 1, 1] + 348*a[1, 1, 0, 1] + 54*a[1, 1, 1, 1]]])

In [18]:
FourthOrderTensor(dim=dim).apply(x)


Matrix([
[98*a[0, 0, 0, 0] + 348*a[0, 0, 0, 1] + 54*a[0, 0, 1, 1], 98*a[0, 0, 0, 1] + 348*a[0, 1, 0, 1] + 54*a[1, 1, 0, 1]],
[98*a[0, 0, 0, 1] + 348*a[0, 1, 0, 1] + 54*a[1, 1, 0, 1], 98*a[0, 0, 1, 1] + 348*a[1, 1, 0, 1] + 54*a[1, 1, 1, 1]]])

In [19]:
def linear_elastic(eps):
    dim = eps.shape[0]
    lmbda, mu = sp.symbols("lambda, mu", positive=True)
    return lmbda * sp.trace(eps) * sp.eye(dim) + 2 * mu * eps

In [21]:
FourthOrderTensor(dim=2).from_operator(linear_elastic).as_voigt()


Matrix([
[lambda + 2*mu,        lambda,  0],
[       lambda, lambda + 2*mu,  0],
[            0,             0, mu]])

In [24]:
def rotation(x, dim = 2):
    r = sp.MatrixSymbol("R", x.shape[0], x.shape[0])  # generic unsymmetric matrix
    if dim == 2:
        assert x.shape[0] == 2
        theta = sp.symbols("theta")
        sin = sp.sin(theta)
        cos = sp.cos(theta)
        r = sp.Matrix([[cos, -sin], [sin, cos]])
    return r.T * x * r


In [25]:
FourthOrderTensor(dim=dim).from_operator(rotation).as_voigt()


Matrix([
[  cos(theta)**2,  sin(theta)**2,  sin(2*theta)/2],
[  sin(theta)**2,  cos(theta)**2, -sin(2*theta)/2],
[-sin(2*theta)/2, sin(2*theta)/2,  cos(2*theta)/2]])

In [36]:

# Define the symbolic coefficients
C11, C12, C22, C16, C26 = sp.symbols('C11 C12 C22 C16 C26')

# Define the 2D fourth-order tensor for orthorhombic crystal
# The stiffness matrix representation in Voigt notation for 2D is:
# [ C11  C12  C16 ]
# [ C12  C22  C26 ]
# [ C16  C26  C66 ]
# Since C66 = (C11 - 2*C12 + C22)/2 for orthorhombic symmetry, we can derive it

C66 = (C11 - 2 * C12 + C22) / 2

# Create the array representation of the tensor
stiffness_matrix = sp.Matrix([
    [C11, C12, C16],
    [C12, C22, C26],
    [C16, C26, C66]
])

# Initialize the FourthOrderTensor using the stiffness matrix
# We need to convert this matrix back to its fourth-order representation
def linear_operator(matrix):
    """
    This function takes a 2D stress/strain matrix and returns the
    result of applying the stiffness matrix.
    """
    C = np.zeros((2, 2, 2, 2), dtype=object)
    C[0, 0, 0, 0] = C11
    C[0, 0, 1, 1] = C12
    C[1, 1, 0, 0] = C12
    C[1, 1, 1, 1] = C22
    C[0, 0, 0, 1] = C16
    C[0, 0, 1, 0] = C16
    C[0, 1, 0, 0] = C16
    C[1, 0, 0, 0] = C16
    C[1, 1, 0, 1] = C26
    C[1, 1, 1, 0] = C26
    C[0, 1, 1, 1] = C26
    C[1, 0, 1, 1] = C26
    C[0, 1, 0, 1] = C66
    C[1, 0, 0, 1] = C66
    C[0, 1, 1, 0] = C66
    C[1, 0, 1, 0] = C66
    return C

# Create the fourth-order tensor
orthorhombic_tensor = FourthOrderTensor(dim=2).from_operator(linear_operator)

# Display the tensor as an array
tensor_array = orthorhombic_tensor.as_array()
print(tensor_array)

AssertionError: 

In [37]:
FourthOrderTensor(dim=2, symmetry='major').as_voigt()


Matrix([
[a[0, 0, 0, 0], a[0, 0, 1, 1], a[0, 0, 0, 1]],
[a[0, 0, 1, 1], a[1, 1, 1, 1], a[1, 1, 0, 1]],
[a[0, 0, 0, 1], a[1, 1, 0, 1], a[0, 1, 0, 1]]])

In [38]:
FourthOrderTensor(dim=2).as_array().as_explicit()


[[[[a[0, 0, 0, 0], a[0, 0, 0, 1]], [a[0, 0, 1, 0], a[0, 0, 1, 1]]], [[a[0, 1, 0, 0], a[0, 1, 0, 1]], [a[0, 1, 1, 0], a[0, 1, 1, 1]]]], [[[a[1, 0, 0, 0], a[1, 0, 0, 1]], [a[1, 0, 1, 0], a[1, 0, 1, 1]]], [[a[1, 1, 0, 0], a[1, 1, 0, 1]], [a[1, 1, 1, 0], a[1, 1, 1, 1]]]]]

In [40]:
a_mandel = sp.MatrixSymbol(dim, dim)


TypeError: MatrixSymbol.__new__() missing 1 required positional argument: 'm'

In [58]:
C11, C12, C22, C16, C26 = sp.symbols('C11 C12 C22 C16 C26')

C1, C2 = sp.symbols('C1 C2', positive=True)
C11 = C22 = C1
C66 = C2

stiffness_matrix_voigt = sp.Matrix([
    [C11, C12, 0],
    [C12, C22, 0],
    [0, 0, C66]
])

In [59]:
a = FourthOrderTensor(dim).from_voigt(stiffness_matrix_voigt)  # initialize from Voigt notation
a.as_voigt()

Matrix([
[ C1, C12,  0],
[C12,  C1,  0],
[  0,   0, C2]])

In [70]:
a.as_voigt()

Matrix([
[ C1, C12,  0],
[C12,  C1,  0],
[  0,   0, C2]])